# Ceballos-inspired LIF: compare passive parameter mappings

Explore how leak and coupling change soma–dendrite voltage, axial current, and spiking. This notebook now uses the local `two_compartment.py` simulator, without Brian2. Parameter attributions below are inherited notes and still require source verification.


In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'two_compartment.py').exists():
    root = root / 'CA1_reduced_LIF'
if not (root / 'two_compartment.py').exists():
    raise FileNotFoundError('Open from the repository root or CA1_reduced_LIF folder')
sys.path.insert(0, str(root.resolve()))


# Ceballos-inspired soma-dendrite LIF baseline

One cell, two electrically coupled compartments. Start with passive loading and threshold/reset; add ionic dynamics only after validating this baseline.

The separate PDF contains the full parameter comparison.

Sources: [article](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1013126) and [HHca notebook](https://github.com/rodrigo-pena-lab/interleaved_resonance/blob/main/code.ipynb).

## Parameter provenance

Conductance densities are mS/cm². Article values below are Methods defaults; figure-specific values can differ.

| Parameter | Symbol | HHca | Article Methods |
| --- | --- | --- | --- |
| Capacitance density | C | 1 uF/cm2 | 1 uF/cm2 |
| Leak conductance | Gl / gL | 0.05 | 0.04 |
| Transient sodium | Gna / gNaT | 35 | 35 |
| Persistent sodium | Gnap / gNaP | 0.3 | 0.9 |
| Delayed rectifier K | Gkdr / gKdr | 6 | 5.8 |
| A-type K | Ga / gKA | 1.4 | 2.8 |
| M-type K | Gm / gKM | 1 | 2.4 |
| High-threshold calcium | Gca / gHCa | 0 | 0.17 |
| Fast Ca-activated K | Gc / gfKCa | 10 | 9.05 |
| Slow Ca-activated K | Gsahp / gsKCa | 5 | 2.8 |
| Dendritic h current | gh | Not implemented | 8.46 |
| Axial coupling | gac | Not implemented | 0.2 |
| Leak reversal | Vl / EL | -70 mV | -70 mV |
| Sodium reversal | Vna / ENa | 55 mV | 55 mV |
| Potassium reversal | Vk / EK | -90 mV | -90 mV |
| Calcium reversal | Vca / ECa | 120 mV | 120 mV |

## Reduction assumptions

| Parameter | Your original | Article passive mapping |
| --- | --- | --- |
| Effective area per compartment | Implicit | 20,000 um2 (chosen) |
| Cs = Cd | 200 pF | 200 pF |
| gLs = gLd | 10 nS | 8 nS |
| g_ax | 10 nS | 40 nS |
| Isolated passive C / gL | 20 ms | 25 ms |
| EL | -70 mV | -70 mV |
| Threshold / reset | -50 / -65 mV | -50 / -65 mV (chosen) |
| Somatic step, 100-400 ms | 400 pA | 400 pA (retained choice) |

Each compartment has an assumed effective area of 20,000 µm². The article coupling density maps to one 40 nS link using this reference area. `HHca_leak` combines the notebook leak with the article axial coupling; it is a hybrid. Threshold, reset, and the current step remain your modeling choices. No refractory period is imposed, matching your original model.

In [ ]:
import matplotlib.pyplot as plt


## Equations

$$C_s\dot v_s=g_{Ls}(E_L-v_s)+I_s+I_{ax}$$
$$C_d\dot v_d=g_{Ld}(E_L-v_d)+I_d-I_{ax}$$
$$I_{ax}=g_{ax}(v_d-v_s)$$

Only the soma resets. The dendrite retains its voltage. Forward Euler updates both voltages from the same previous state, with a 0.025 ms step; threshold detection remains on the time grid. The step current is held constant within each time step.

In [ ]:
from two_compartment import simulate


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True, sharey='row')
for col, mode in enumerate(('original', 'paper', 'HHca_leak')):
    rows, spikes = simulate(mode)
    t, vs, vd, axial, Is, Id = zip(*rows)
    axes[0, col].plot(t, vs, label='Soma')
    axes[0, col].plot(t, vd, label='Dendrite')
    axes[0, col].axhline(-50, color='gray', ls=':', lw=0.8)
    axes[0, col].set_title(f'{mode}: {len(spikes)} spikes')
    axes[0, col].legend()
    axes[1, col].plot(t, axial, color='purple')
    axes[1, col].axhline(0, color='gray', lw=0.8)
    axes[1, col].set_xlabel('Time (ms)')
    print(f'{mode}: {len(spikes)} spikes')
axes[0, 0].set_ylabel('Voltage (mV)')
axes[1, 0].set_ylabel('Axial current into soma (pA)')
fig.suptitle('Passive parameter mappings: somatic 400 pA step, 100–400 ms')
fig.tight_layout()
plt.show()


## Interpret the traces

Negative axial current means the soma is charging the dendrite. After reset, current may reverse as the dendrite supplies charge to the soma. Compare uncoupled, weakly coupled, and strongly coupled cases before adding active currents.

A passive two-compartment system has no intrinsic subthreshold impedance resonance. Frequency-dependent spiking is still possible. This baseline does not preserve the HH ionic mechanisms for bursting or the action-potential waveform.

For your original parameters, the no-reset DC effective somatic conductance is $g_{Ls}+g_{ax}g_{Ld}/(g_{ax}+g_{Ld})=15$ nS, giving a steady-state threshold current of 300 pA. Your 400 pA step is above it.

Next: compare somatic versus dendritic stimulation, then use matched 10 Hz and 40 Hz sinusoidal protocols. Calibrate any added active dynamics against a clearly identified HH configuration. Do not interpret the article axial value as a measured intercellular gap-junction conductance.